<a href="https://colab.research.google.com/github/irAbs174/openshell-notebook/blob/main/openshell_notobook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modern AI Agent Isolation with NVIDIA OpenShell: Complete Architecture & Step-by-Step Hands-On Notebook

**NVIDIA OpenShell** is an open-source, policy-driven runtime designed specifically for autonomous, self-evolving AI agents. Unlike traditional container runtimes that isolate generic workloads, OpenShell provides **out-of-process, zero-trust environmental guardrails**.

An AI agent running inside an OpenShell sandbox can self-evolve, write new Python scripts mid-task, and install tools—yet it remains completely incapable of exfiltrating credentials, traversing unauthorized filesystem paths, or making unapproved outbound network requests.

---

## 1. Deep-Dive Architecture & Core Concepts

OpenShell adopts a security model analogous to a modern web browser's tab sandbox:

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                             HOST ENVIRONMENT                                │
│                                                                             │
│  ┌───────────────────────┐             ┌─────────────────────────────────┐  │
│  │     OpenShell CLI     │             │        openshell-gateway        │  │
│  │  (`openshell sandbox`)│             │   (gRPC / HTTP Server :17670)   │  │
│  └───────────┬───────────┘             └────────────────┬────────────────┘  │
│              │                                          │                   │
│              └──────────────────┐  ┌────────────────────┘                   │
│                                 ▼  ▼                                        │
│  ┌───────────────────────────────────────────────────────────────────────┐  │
│  │                    SANDBOX EXECUTION ENVIRONMENT                      │  │
│  │  ┌─────────────────────────────────────────────────────────────────┐  │  │
│  │  │                        Agent Harness                            │  │  │
│  │  │                 (Claude Code, Codex, Custom)                    │  │  │
│  │  └──────────────────────────────┬──────────────────────────────────┘  │  │
│  │                                 │ (Executes Commands)                 │  │
│  │                                 ▼                                     │  │
│  │  ┌─────────────────────────────────────────────────────────────────┐  │  │
│  │  │                       OUT-OF-PROCESS POLICY                     │  │  │
│  │  ├─────────────────────────────────────────────────────────────────┤  │  │
│  │  │ 1. Filesystem Layer : Linux Landlock Sandboxing                 │  │  │
│  │  │ 2. Network Layer    : L7 Proxy Filtering (Host, API, Egress)    │  │  │
│  │  │ 3. Privacy Router   : Zero-Trust LLM Credential Swapping        │  │  │
│  │  └─────────────────────────────────────────────────────────────────┘  │  │
│  └───────────────────────────────────────────────────────────────────────┘  │  │
└─────────────────────────────────────────────────────────────────────────────┘

```

### Key Architectural Pillars

1. **Out-of-Process Enforcement**: Security controls live *outside* the agent's context window and process boundary. Even if an attacker executes a successful prompt injection against the agent, the kernel and OpenShell gateway refuse prohibited syscalls and network routes.
2. **Linux Landlock Filesystem Isolation**: Enforces granular directory-level read/write rules at the Linux kernel level.
3. **L7 Proxy & Network Filtering**: Outbound HTTP/HTTPS requests pass through a transparent policy proxy. Unapproved endpoints are immediately blocked with `403` status codes.
4. **Privacy Router (Credential Isolation)**: Keeps LLM provider API keys outside the sandbox entirely. The agent directs local inference requests to the internal route, and OpenShell swaps in authorized backend API credentials in-flight.

---

## 2. High-Performance Interactive Jupyter Notebook

Copy and paste the code cells below directly into a `.ipynb` notebook. Executing the cells sequentially will take you from zero to running an isolated agent with custom declarative policies.

---

### Markdown Cell 1: Environment Setup & Diagnostics

```markdown
# Section 1: OpenShell System Diagnostics & Environment Verification
In this section, we verify the host environment, check available compute drivers (Docker, Podman, or Kubernetes), and inspect OpenShell binaries.

```

### Code Cell 1

In [ ]:
import os
import shutil
import subprocess
import sys


def run_command(cmd, verbose=True):
  """Executes shell commands and streams output safely inside Jupyter."""
  print(f"\033[1;34m[EXEC]\033[0m {cmd}")
  process = subprocess.Popen(
      cmd,
      shell=True,
      stdout=subprocess.PIPE,
      stderr=subprocess.PIPE,
      text=True,
  )

  stdout_lines, stderr_lines = [], []
  while True:
    output = process.stdout.readline()
    if output == "" and process.poll() is not None:
      break
    if output and verbose:
      print(output.strip())
      stdout_lines.append(output)

  _, stderr = process.communicate()
  if stderr and verbose:
    print(f"\033[1;31m[STDERR]\033[0m\n{stderr.strip()}")

  return process.returncode, "".join(stdout_lines), stderr


# 1. Verify system tools
tools = ["openshell", "openshell-gateway", "docker", "curl"]
for tool in tools:
  path = shutil.which(tool)
  status = f"\033[1;32mFOUND\033[0m at {path}" if path else "\033[1;31mNOT FOUND\033[0m"
  print(f"Tool check [{tool}]: {status}")

# 2. Check OpenShell versions
run_command("openshell --version")
run_command("openshell-gateway --version")

---

### Markdown Cell 2: Launching and Configuring `openshell-gateway`

```markdown
# Section 2: Start the OpenShell Gateway Service
`openshell-gateway` is the background gRPC/HTTP daemon managing sandbox life-cycles, mTLS client certificates, and policy evaluations.

```

### Code Cell 2

In [ ]:
import time

# Define environment configuration for local development gateway
gateway_port = 17670
gateway_env = os.environ.copy()
gateway_env["OPENSHELL_BIND_ADDRESS"] = "127.0.0.1"
gateway_env["OPENSHELL_SERVER_PORT"] = str(gateway_port)
gateway_env["OPENSHELL_LOG_LEVEL"] = "info"
gateway_env["OPENSHELL_DISABLE_TLS"] = "true"  # Local development mode

print("Starting openshell-gateway in background...")
gateway_proc = subprocess.Popen(
    [
        "openshell-gateway",
        "--bind-address",
        "127.0.0.1",
        "--port",
        str(gateway_port),
        "--disable-tls",
        "--log-level",
        "info",
    ],
    env=gateway_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

# Wait for gateway initialization
time.sleep(3)

# Test gateway connection using CLI doctor/status
code, out, err = run_command(
    f"openshell --gateway-endpoint http://127.0.0.1:{gateway_port} gateway"
    " doctor"
)

# Register local gateway into metadata configuration
run_command(
    f"openshell gateway add http://127.0.0.1:{gateway_port} --local"
    " --name local-dev"
)

---

### Markdown Cell 3: Creating and Inspecting a Sandbox

```markdown
# Section 3: Sandbox Lifecycle Management
We will create an isolated execution sandbox and examine default security constraints.

```

### Code Cell 3

In [ ]:
# 1. Create a isolated sandbox named 'agent-demo'
run_command("openshell sandbox create --name agent-demo")

# 2. List active sandboxes
run_command("openshell sandbox list")

# 3. Test outbound network reachability inside the default sandbox
# Outbound connections should be rejected by the L7 policy engine by default.
test_net_cmd = (
    "openshell exec agent-demo -- curl -sS --connect-timeout 5"
    " https://api.github.com/zen"
)
run_command(test_net_cmd)

---

### Markdown Cell 4: Writing & Applying Declarative Policies

```markdown
# Section 4: Dynamic Policy Injection
OpenShell policies govern Filesystem, Network L7 pathing, and Process execution without needing to restart the sandbox.

```

### Code Cell 4

In [ ]:
# Create a strict policy YAML file
policy_content = """
apiVersion: v1alpha1
kind: SandboxPolicy
metadata:
  name: demo-github-read-only
spec:
  network:
    egress:
      - match:
          host: "api.github.com"
          scheme: "https"
        rules:
          - methods: ["GET"]
            path: "/*"
            action: Allow
          - methods: ["POST", "PUT", "DELETE"]
            path: "/*"
            action: Deny
  filesystem:
    readOnlyPaths:
      - "/usr"
      - "/lib"
    readWritePaths:
      - "/tmp"
      - "/workspace"
  process:
    allowExecution:
      - "/bin/*"
      - "/usr/bin/*"
"""

policy_filename = "github_policy.yaml"
with open(policy_filename, "w") as f:
  f.write(policy_content.strip())

print(f"Created policy file: {policy_filename}")

# Apply policy dynamically to the running sandbox
run_command(f"openshell policy set agent-demo --policy {policy_filename} --wait")

---

### Markdown Cell 5: Verifying Security Controls (Egress & Method Tests)

```markdown
# Section 5: Verify Policy Enforcement
We test HTTP GET (Allowed) vs HTTP POST (Denied) on `api.github.com`.

```

### Code Cell 5

In [ ]:
print("--- TEST 1: HTTP GET (Should succeed) ---")
run_command(
    "openshell exec agent-demo -- curl -sS -X GET https://api.github.com/zen"
)

print("\n--- TEST 2: HTTP POST (Should be BLOCKED by L7 proxy) ---")
run_command(
    "openshell exec agent-demo -- curl -sS -X POST"
    " https://api.github.com/user/repos -d '{\"name\":\"exploit\"}'"
)

print("\n--- TEST 3: Unallowed Egress Host (Should be BLOCKED) ---")
run_command(
    "openshell exec agent-demo -- curl -sS --connect-timeout 3"
    " https://example.com"
)

---

### Markdown Cell 6: Inspecting Sandbox Audit Logs & Teardown

```markdown
# Section 6: Security Audit Logging & Sandbox Cleanup
OpenShell logs every intercepted request, policy violation, and sandbox event.

```

### Code Cell 6

In [ ]:
# Fetch security logs
print("=== SANDBOX AUDIT LOGS ===")
run_command("openshell logs agent-demo --tail 20")

# Teardown sandbox resources
print("\n=== CLEANUP ===")
run_command("openshell sandbox delete agent-demo --force")

# Stop gateway process
if 'gateway_proc' in locals():
  gateway_proc.terminate()
  print("OpenShell Gateway server stopped successfully.")

---

## 3. CLI & Gateway Command Quick Reference

| OpenShell Subcommand | Description |
| --- | --- |
| `openshell sandbox create` | Spawns a new isolated execution sandbox |
| `openshell policy set <name> -p <file>` | Applies hot-reloaded YAML policies to a sandbox |
| `openshell logs <name>` | Streams real-time network and filesystem audit logs |
| `openshell-gateway --config <file>` | Boots the central runtime gateway daemon |
| `openshell-gateway generate-certs` | Generates PKI certificates for mTLS auth |